In [7]:
import numpy as np  # importation du numpy pour l'outils mathématiques

# ============================================================
# FONCTIONS UTILITAIRES
# ============================================================

def mse(y, y_pred):
    # L'Erreur Quadratique Moyenne (MSE).
    return np.mean((y - y_pred) ** 2)

def sigmoid(z):
    # La fonction magique qui écrase n'importe quel nombre pour le faire rentrer
    # entre 0 et 1. C'est parfait pour transformer des valeurs en probabilités.
    return 1 / (1 + np.exp(-z))

def bce(y, p):
    # l'outil de mesure d'erreur pour la classification binaire.
    eps = 1e-12  # Une toute petite marge de sécurité pour ne jamais faire log(0) qui ferait planter le code.
    p = np.clip(p, eps, 1 - eps)  # On bride les probabilités entre 0.000...1 et 0.999...9
    return -np.mean(y * np.log(p) + (1 - y) * np.log(1 - p))

# ============================================================
# PARTIE A) (FROM SCRATCH)
# ============================================================

def regression_lineare_from_scratch():
    # Nos données d'entraînement : un petit jeu de données  simple à 1 dimension.
    X = np.array([1, 2, 3, 4, 5], dtype=float)
    y = np.array([2, 4, 5, 4, 5], dtype=float)

    # On initialise notre pente (a) et notre point de départ (b0) à zéro.
    a, b0 = 0.0, 0.0

    lr = 0.01          # Le taux d'apprentissage (la taille de nos pas à chaque itération).
    epochs = 200000    # Le nombre maximum d'essais qu'on se donne pour apprendre.
    epsilon = 1e-10    # Notre seuil. Si l'erreur change moins que ça, on s'arrête.

    previous_cost = float("inf") # Au début, notre erreur est infinie.

    for i in range(epochs):
        # 1. On fait une prédiction avec notre modèle actuel (y = ax + b)
        y_pred = a * X + b0
        # 2. On évalue à quel point on s'est trompé
        current_cost = mse(y, y_pred)

        # Sécurité : si l'erreur explose (devient infinie), c'est que notre "lr" est trop grand.
        if not np.isfinite(current_cost):
            print("\n Régression Linéaire (simple)")
            print(" L'algorithme a divergé. Essayez un lr plus petit ")
            return None

        # Si l'amélioration est petit, on arrête de faire des calculs pour rien.
        if abs(previous_cost - current_cost) < epsilon:
            break

        previous_cost = current_cost

        # 3. On calcule les gradients (la pente de l'erreur) pour savoir dans quelle direction corriger 'a' et 'b0'.
        da = (-2 / len(X)) * np.sum(X * (y - y_pred))
        db = (-2 / len(X)) * np.sum(y - y_pred)

        # 4. On met à jour nos paramètres en faisant un petit pas dans le sens inverse de l'erreur.
        a = a - lr * da
        b0 = b0 - lr * db

    print("\n Régression Linéaire (simple)")
    print("Nombre d'itérations =", i + 1)
    print("Erreur finale (MSE) =", float(current_cost))
    print(f"Modèle trouvé       = y = {a:.6f}*x + {b0:.6f}")

    return a, b0, current_cost, i + 1


def regression_multiple_from_scratch():
    # Ici, on a deux caractéristiques (colonnes) pour prédire 'y'.
    X = np.array([
        [1, 2],
        [2, 0],
        [3, 1],
        [4, 3],
        [5, 4]
    ], dtype=float)

    y = np.array([3, 3, 4, 7, 9], dtype=float)


    # On met toutes les données à la même échelle pour éviter que l'algorithme ne s'affole.
    X_mean = X.mean(axis=0)
    X_std = X.std(axis=0) + 1e-12 # + 1e-12 pour éviter de diviser par zéro si l'écart-type est nul
    Xn = (X - X_mean) / X_std

    n, d = Xn.shape # n = nombre de lignes, d = nombre de colonnes (features)
    w = np.zeros(d) # Nos poids (un par caractéristique)
    b0 = 0.0        # Notre biais (l'ordonnée à l'origine)

    lr = 0.05
    epochs = 200000
    epsilon = 1e-10

    previous_cost = float("inf")

    for i in range(epochs):
        # Produit matriciel : calcule les prédictions pour toutes les lignes d'un coup
        y_pred = np.dot(Xn, w) + b0
        current_cost = mse(y, y_pred)

        if not np.isfinite(current_cost):
            print("\n Régression Linéaire (multiple)")
            print(" L'algorithme a divergé.")
            return None

        # Arrêt anticipé si on ne progresse plus.
        if abs(previous_cost - current_cost) < epsilon:
            break

        previous_cost = current_cost

        # Calcul des gradients version matricielle
        dw = (-2 / n) * np.dot(Xn.T, (y - y_pred))
        db = (-2 / n) * np.sum(y - y_pred)

        # Mise à jour
        w = w - lr * dw
        b0 = b0 - lr * db

    print("\n Régression Linéaire (multiple)")
    print("Nombre d'itérations =", i + 1)
    print("Erreur finale (MSE) =", float(current_cost))
    print("Poids (w)           =", w)
    print("Biais (b)           =", float(b0))

    return w, b0, current_cost, i + 1


def regression_polynomial_from_scratch():
    x = np.array([0, 1, 2, 3, 4, 5], dtype=float)
    y = np.array([1, 2, 5, 10, 17, 26], dtype=float)

    degree = 2 # On veut trouver une courbe de type ax^2 + bx + c

    # Comme pour le modèle multiple, la normalisation est indispensable
    x_mean = x.mean()
    x_std = x.std() + 1e-12
    xn = (x - x_mean) / x_std

    # On transforme notre unique variable 'x' en plusieurs variables (x^0, x^1, x^2)
    X_poly = np.column_stack([xn ** k for k in range(degree + 1)])

    n, d = X_poly.shape
    w = np.zeros(d) # On initialise tous les poids de notre polynôme à zéro.

    lr = 0.05
    epochs = 200000
    epsilon = 1e-12

    previous_cost = float("inf")

    for i in range(epochs):
        y_pred = np.dot(X_poly, w)
        current_cost = mse(y, y_pred)

        if not np.isfinite(current_cost):
            print("\nRégression Polynomiale")
            print(" L'algorithme a divergé.")
            return None

        if abs(previous_cost - current_cost) < epsilon:
            break

        previous_cost = current_cost

        # Mise à jour des poids via la descente de gradient
        dw = (-2 / n) * np.dot(X_poly.T, (y - y_pred))
        w = w - lr * dw

    print("\n Régression Polynomiale")
    print("Degré               =", degree)
    print("Nombre d'itérations =", i + 1)
    print("Erreur finale (MSE) =", float(current_cost))
    print("Poids (w)           =", w)
    return w, current_cost, i + 1


def logistique_from_scratch():
    # Ici, on ne veut plus prédire une valeur, mais une catégorie (0 ou 1).
    X = np.array([0, 1, 2, 3, 4, 5], dtype=float)
    y = np.array([0, 0, 0, 1, 1, 1], dtype=float)

    a, b0 = 0.0, 0.0
    n = len(X)

    lr = 0.1
    epochs = 200000
    epsilon = 1e-10

    previous_cost = float("inf")

    for i in range(epochs):
        # On calcule une combinaison linéaire classique...
        z = a * X + b0
        # ... puis on l'écrase entre 0 et 1 avec la sigmoïde pour obtenir une probabilité.
        p = sigmoid(z)

        # On évalue l'erreur avec l'Entropie Croisée Binaire (adaptée aux probabilités)
        current_cost = bce(y, p)

        if not np.isfinite(current_cost):
            print("\n Régression Logistique")
            print("Divergence. Essayez un lr plus petit.")
            return None

        if abs(previous_cost - current_cost) < epsilon:
            break

        previous_cost = current_cost

        # Le calcul du gradient est très élégant car les dérivées se simplifient pour ressembler à celles de la régression linéaire !
        da = (-1 / n) * np.sum(X * (y - p))
        db = (-1 / n) * np.sum(y - p)

        a = a - lr * da
        b0 = b0 - lr * db

    # Évaluation finale du modèle
    p_final = sigmoid(a * X + b0)
    # Règle de décision : si la proba est >= 50%, c'est la classe 1, sinon c'est 0.
    y_pred = (p_final >= 0.5).astype(float)
    acc = float(np.mean(y_pred == y)) # Pourcentage de bonnes réponses

    print("\n Régression Logistique")
    print("Nombre d'itérations =", i + 1)
    print("Erreur finale (BCE) =", float(current_cost))
    print(f"Modèle trouvé       = p = sigmoid({a:.6f}*x + {b0:.6f})")
    print("Précision (Train)   =", acc * 100, "%")

    # Petit test de prédiction
    x_new = 2.5
    p_new = float(sigmoid(a * x_new + b0))
    pred_new = int(p_new >= 0.5)
    print("Prédiction pour x=2.5 -> Proba =", p_new, "Classe =", pred_new)

    return a, b0, current_cost, acc, i + 1

# ============================================================
# PARTIE B) AVEC LA LIBRAIRIE (SCIKIT-LEARN)
# ============================================================

def lib_linear_simple():
    from sklearn.linear_model import LinearRegression

    # scikit-learn attend toujours X sous forme de matrice (colonne), d'où le .reshape(-1, 1)
    X = np.array([1, 2, 3, 4, 5], dtype=float).reshape(-1, 1)
    y = np.array([2, 4, 5, 4, 5], dtype=float)

    # 1. On crée le modèle
    model = LinearRegression()
    # 2. On l'entraîne (il trouve les meilleurs paramètres 'a' et 'b' instantanément)
    model.fit(X, y)

    y_pred = model.predict(X)
    cost = mse(y, y_pred)

    print("\n[LIBRAIRIE] Régression Linéaire (simple)")
    print("Coefficient (a) =", float(model.coef_[0]))
    print("Biais (b)       =", float(model.intercept_))
    print("Erreur (MSE)    =", float(cost))


def lib_linear_multiple():
    from sklearn.linear_model import LinearRegression

    X = np.array([[1, 2], [2, 0], [3, 1], [4, 3], [5, 4]], dtype=float)
    y = np.array([3, 3, 4, 7, 9], dtype=float)

    model = LinearRegression()
    model.fit(X, y)

    y_pred = model.predict(X)
    cost = mse(y, y_pred)

    print("\n[LIBRAIRIE] Régression Linéaire (multiple)")
    print("Coefficients (w) =", model.coef_)
    print("Biais (b)        =", float(model.intercept_))
    print("Erreur (MSE)     =", float(cost))


def lib_polynomial():
    from sklearn.preprocessing import PolynomialFeatures
    from sklearn.linear_model import LinearRegression
    from sklearn.pipeline import Pipeline

    x = np.array([0, 1, 2, 3, 4, 5], dtype=float).reshape(-1, 1)
    y = np.array([1, 2, 5, 10, 17, 26], dtype=float)

    degree = 2

    # Un "Pipeline" permet d'enchaîner les étapes proprement :
    # D'abord on crée les x^2 (PolynomialFeatures), ensuite on fait la régression (LinearRegression).
    model = Pipeline([
        ("poly", PolynomialFeatures(degree=degree, include_bias=True)),
        ("linreg", LinearRegression())
    ])

    model.fit(x, y)

    y_pred = model.predict(x)
    cost = mse(y, y_pred)

    lin = model.named_steps["linreg"]

    print("\n[LIBRAIRIE] Régression Polynomiale")
    print("Degré            =", degree)
    print("Coefficients (w) =", lin.coef_)
    print("Biais (b)        =", float(lin.intercept_))
    print("Erreur (MSE)     =", float(cost))


def lib_logistic():
    from sklearn.linear_model import LogisticRegression

    X = np.array([0, 1, 2, 3, 4, 5], dtype=float).reshape(-1, 1)
    y = np.array([0, 0, 0, 1, 1, 1], dtype=int)

    # penalty=None pour désactiver la régularisation et obtenir les mêmes résultats que notre code
    model = LogisticRegression(penalty=None, solver="lbfgs", max_iter=1000)
    model.fit(X, y)

    # predict_proba renvoie les probabilités pour la classe 0 ET la classe 1. On garde seulement la colonne 1.
    proba = model.predict_proba(X)[:, 1]
    cost = bce(y.astype(float), proba)
    acc = float(np.mean(model.predict(X) == y))

    print("\n[LIBRAIRIE] Régression Logistique (sans régularisation)")
    print("Coefficient (a) =", model.coef_)
    print("Biais (b)       =", model.intercept_)
    print("Erreur (BCE)    =", float(cost))
    print("Précision       =", acc * 100, "%")

    x_new = np.array([[2.5]], dtype=float)
    p_new = float(model.predict_proba(x_new)[0, 1])
    pred_new = int(model.predict(x_new)[0])
    print("Prédiction pour x=2.5 -> Proba =", p_new, "Classe =", pred_new)

# ============================================================
# BLOC MAIN (Exécution)
# ============================================================

if __name__ == "__main__":
    print("===  LIBRAIRIE (RÉSULTATS FINAUX) ===")

    # Exécution de nos implémentations "from scratch"
    regression_lineare_from_scratch()
    regression_multiple_from_scratch()
    regression_polynomial_from_scratch()
    logistique_from_scratch()

    print("\n" + "="*60 + "\n")

    # Exécution des implémentations scikit-learn
    lib_linear_simple()
    lib_linear_multiple()
    lib_polynomial()
    lib_logistic()

===  LIBRAIRIE (RÉSULTATS FINAUX) ===

 Régression Linéaire (simple)
Nombre d'itérations = 2600
Erreur finale (MSE) = 0.48000001469604936
Modèle trouvé       = y = 0.600079*x + 2.199716

 Régression Linéaire (multiple)
Nombre d'itérations = 278
Erreur finale (MSE) = 0.07372549176073687
Poids (w)           = [1.52508121 1.05377882]
Biais (b)           = 5.1999999999989015

 Régression Polynomiale
Degré               = 2
Nombre d'itérations = 436
Erreur finale (MSE) = 1.564478129398118e-11
Poids (w)           = [7.24999409 8.53912564 2.9166708 ]

 Régression Logistique
Nombre d'itérations = 200000
Erreur finale (BCE) = 0.0014564588950867127
Modèle trouvé       = p = sigmoid(10.892257*x + -27.055725)
Précision (Train)   = 100.0 %
Prédiction pour x=2.5 -> Proba = 0.5436185231350762 Classe = 1



[LIBRAIRIE] Régression Linéaire (simple)
Coefficient (a) = 0.6
Biais (b)       = 2.2
Erreur (MSE)    = 0.47999999999999987

[LIBRAIRIE] Régression Linéaire (multiple)
Coefficients (w) = [1.07843137

C:\Users\PC\AppData\Roaming\Python\Python313\site-packages\sklearn\linear_model\_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(
